**Richard Davies, Finn McEcoy, Josh Hellings**  - Automated Data Visualisation for Policymaking 2026/27

# Transforming Data

The bread and butter of a data workflow is cleaning and preparation, taking raw datasets and transforming them into a useful form.

</br></br></br></br>



## Introducing Tools: Pandas

The first tool we'll use today is `Pandas`, a Python library used to work with datasets. It provides access to `DataFrames` - tables we analyse with code.

Python already has a few built in data structures, for example lists and dictionaries:

</br></br></br>

First, let's import the Pandas package.

In [ ]:
import pandas as pd     # using 'as' just allows us to use a shorthand for accessing package functions. e.g. pd.read_csv() instead of pandas.read_csv()

In [ ]:
london = {
    "name": "London",
    "population": 8308369,
    "area": 1572
} # This is an example of a dictionary

locations = [ # This is an example of a list of dictionaries
    {
        "name": "London",
        "population": 8_982_000,
        "area": 606
    },
    {
        "name": "Newport",
        "population": 128_060,
        "area": 32.52
    },
    {
        "name": "Darlington",
        "population": 93_015,
        "area": 7.62
    },

]


<br>
<br>
<br>
<br>
Which we can turn into Pandas `DataFrames`

In [ ]:
df = pd.DataFrame(locations)
df

<br>

We can access column values using the column name

In [ ]:
df['area']

<br>

Just like we can perform mathematical operations on variables in Python, we can perform operations on entire columns.

For example, we can add a density column:

In [ ]:
df['density'] = df['population'] / df['area']
df

</br></br>
</br>
</br>
</br>
or sort our dataframe

In [ ]:
sorted_df = df.sort_values(by=['density'], ascending=False)
sorted_df

<br>

# Practical: Transforming data with `Pandas`

First, we'll use a built-in Pandas function for reading Excel files. Pandas also contains functions for reading other types of files (e.g. `.read_csv()`, `.read_json()`, `.read_html`, etc. Find them all [here](https://pandas.pydata.org/docs/reference/io.html).)

When reading in a file, we can either use a *relative* link to access files stored locally, or we can pass a URL to automatically read in files from the Web. Here, we'll use the raw link to an Excel file stored on GitHub.

- **Note**: A Pandas DataFrame represents just one table, whereas Excel files can have multiple tables (sheets), so when using `.read_excel()`, we should specify the sheet_name we want to read in.

In [ ]:
df = pd.read_excel("https://raw.githubusercontent.com/jhellingsdata/RADataHub/refs/heads/main/misc/consumertrendsq22024cpsa1.xlsx", sheet_name="0GSCS") # Read the data from the Excel file, specifying the sheet name and skipping the first 5 rows

df.head(10)

<br>

We're getting the data, but there are lots of rows we don't need before we get to our variable names. To fix this, we can use the `skiprows` parameter.

In [ ]:
df = pd.read_excel("https://raw.githubusercontent.com/jhellingsdata/RADataHub/refs/heads/main/misc/consumertrendsq22024cpsa1.xlsx", sheet_name="0GSCS", skiprows=5) # Read the data from the Excel file, specifying the sheet name and skipping the first 5 rows

# Let's also make the column names more readable
df = df.rename(columns={
    'Time period and codes': 'date',
    'Total goods': 'Goods'
})

df.head(10)

Now, we've got our series names as column headers, but there's a couple rows we don't need at the top. There are lots of ways we could remove these. We'll do it by removing any rows where the `date` value is non-numeric.

In [ ]:
# Let's make the date column a number and drop everything that isn't one
df['date'] = pd.to_numeric(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
# Then convert it back to an integer (to make sure we can use it as a date (temporal) later in Vega-Lite)
df['date'] = df['date'].astype(int)
df.head()

Now, let's select just the columnd we want. We can do this using double square brackets, where the inner list is just a list of the columns we want to keep.

In [ ]:
# We only care about goods and services. Let's keep those
df = df[['date', 'Goods', 'Services']]
df.head(10)

Almost there! Lastly we need to transform the data into long-format. This is the preferred data format for many visualisation languages, including Vega-Lite.

**What is long-form?** Long-form data is when each row represents a single observation. In this case, each row would represent a single data point for a single date and series (Goods or Services).

In [ ]:
# Let's make this data long-form using Panda's melt function
df = df.melt(id_vars=['date'], var_name='series', value_name='value')   # id_vars are the columns we want to keep as they are, var_name is the name of the column that will contain the old column names, and value_name is the name of the column that will contain the old values

# And save it
df.to_csv("consumertrendsq22024cpsa1_long.csv", index=False)
df

Now, we could upload this to GitHub and use it to make a chart in Vega-Lite.

<br>

---